In [37]:
import os
import psycopg2
import pandas as pd

os.environ['PYTHONUTF8'] = '1'

# Cambia 'events' por el nombre real de tu tabla
table_name = 'events'

query = f"""
SELECT 
    c.column_name AS columna_bd,
    c.data_type AS tipo_dato,
    CASE 
        WHEN c.data_type IN ('integer', 'bigint', 'smallint') THEN 'int'
        WHEN c.data_type LIKE '%char%' OR c.data_type = 'text' THEN 'str'
        WHEN c.data_type = 'boolean' THEN 'bool'
        WHEN c.data_type LIKE '%timestamp%' OR c.data_type = 'date' THEN 'datetime'
        ELSE c.data_type
    END AS tipo_python
FROM information_schema.columns c
WHERE c.table_schema = 'shared'
  AND c.table_name = '{table_name}'
ORDER BY c.ordinal_position;
"""

try:
    conn = psycopg2.connect(
        host="localhost",
        database="postgres",
        user="postgres",
        password="1234",
        options="-c client_encoding=UTF8"
    )
    
    df_map = pd.read_sql(query, conn)
    print("✅ Columnas de la base de datos:")
    print(df_map.to_string(index=False))
    
except Exception as e:
    print(f"❌ Error: {e}")
finally:
    if 'conn' in locals():
        conn.close()

✅ Columnas de la base de datos:
Empty DataFrame
Columns: [columna_bd, tipo_dato, tipo_python]
Index: []


C:\Users\detla\AppData\Local\Temp\ipykernel_8532\2908150144.py:36: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_map = pd.read_sql(query, conn)


In [41]:
import os
import psycopg2
import pandas as pd
from datetime import datetime
from psycopg2.extras import execute_values

os.environ['PYTHONUTF8'] = '1'

# Cargar CSV
df = pd.read_csv('data/eventos.csv')  # Cambia por tu ruta real

# Mapeo de columnas
column_mapping = {
    'id_Kulturklik': 'external_id',
    'type': 'type',
    'typeEs': 'subtipo',
    'nameEs': 'establishment'
}

df_mapped = df.rename(columns=column_mapping)

# Campos con valores por defecto
df_mapped['municipality_id'] = None  
df_mapped['start_date'] = None       
df_mapped['end_date'] = None         
df_mapped['publication_date'] = datetime.now()
df_mapped['price_eur'] = 0.0
df_mapped['is_free'] = True
df_mapped['is_sponsored'] = False
df_mapped['online'] = False
df_mapped['active'] = True
df_mapped['created_at'] = datetime.now()
df_mapped['language'] = 'es'

# Columnas que existen en la BD (ajusta si faltan o sobran según tu tabla real)
columns_bd = [
    'external_id', 'type', 'subtipo', 'establishment',
    'municipality_id', 'start_date', 'end_date', 'publication_date',
    'price_eur', 'is_free', 'is_sponsored', 'online', 'active',
    'created_at', 'language'
]

# Asegurar que solo seleccionamos columnas que existen en el dataframe
cols_existentes = [c for c in columns_bd if c in df_mapped.columns]
df_insert = df_mapped[cols_existentes].copy()

# Eliminar duplicados por external_id
if 'external_id' in df_insert.columns:
    df_insert = df_insert.drop_duplicates(subset=['external_id'], keep='first')

try:
    conn = psycopg2.connect(
        host="localhost",
        database="postgres",
        user="postgres",
        password="1234",
        options="-c client_encoding=UTF8"
    )
    
    cursor = conn.cursor()
    
    columns_str = ', '.join(cols_existentes)
    query = f"""
        INSERT INTO market_data.events ({columns_str})
        VALUES %s
        ON CONFLICT (external_id) DO NOTHING
    """
    
    data_tuples = [tuple(row) for row in df_insert.itertuples(index=False, name=None)]
    execute_values(cursor, query, data_tuples)
    
    conn.commit()
    print(f"✅ Insertadas {len(data_tuples)} filas procesadas")
    
except Exception as e:
    print(f"❌ Error: {e}")
    if 'conn' in locals():
        conn.rollback()
finally:
    if 'conn' in locals():
        conn.close()

❌ Error: INSERT tiene más expresiones que columnas de destino
LINE 3: ...alse,true,'2026-06-05T13:57:12.151419'::timestamp,'es'),(202...
                                                             ^

